# Select DESI DR2 loa spectra for a VAE

Build a reusable catalog selection **before reading the spectral arrays**. This notebook saves `loa_good.sqlite`, which `read_loa_selected.py extract` can consume directly. It is self-contained for selection; no helper Python file is required.

Run in NERSC Jupyter with a compute-backed session and a kernel containing `numpy` and `fitsio`. The catalog has roughly 64 million rows; the notebook reads 200,000 rows at a time. Allow time and disk space for the index. This notebook has not been run against NERSC data locally; results appear when you run it.

Default cuts: primary spectrum; Redrock class GALAXY or QSO; ZWARN=0; COADD_FIBERSTATUS=0; positive TARGETID; finite redshift greater than -1. These are configurable baseline cuts, not a complete tracer-specific DESI science selection or a guarantee of correct QSO classifications.

**Pixel validity and S/N are not evaluated here.** The extraction script applies the separate valid-pixel fraction cut after reading spectra.


## 1. Check the Python kernel


In [1]:
import sys
import argparse
import json
import os
from pathlib import Path
import sqlite3
import uuid

import numpy as np
import fitsio
from IPython.display import display, Markdown

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("fitsio:", fitsio.__version__)


Python: /global/common/software/nersc/pe/conda-envs/26.8.1/python-3.13/nersc-python/bin/python
NumPy: 2.5.2
fitsio: 1.4.2


## 2. Configure the selection
Keep the catalog version explicit. All surveys and programs are included unless you set `SURVEYS`. Use a new output index name when changing cuts.


In [9]:
CATALOG = Path("/global/cfs/cdirs/desi/spectro/redux/loa/zcatalog/v1/zall-pix-loa.fits")
SELECTION = Path("/pscratch/sd/z/zixinq/loa_selected.sqlite")
TYPES = ["GALAXY", "QSO", "STAR"]
SURVEYS = None  # e.g. ["main"] to restrict survey coverage
MIN_DELTACHI2 = None  # optional extra threshold; no universal threshold assumed
CHUNK_ROWS = 200_000

COLUMNS = ["TARGETID", "SURVEY", "PROGRAM", "HEALPIX", "ZCAT_PRIMARY",
           "SPECTYPE", "ZWARN", "COADD_FIBERSTATUS", "Z", "ZERR", "DELTACHI2"]
assert CHUNK_ROWS > 0
assert CATALOG.is_file(), f"Catalog not found: {CATALOG}"
print("Catalog:", CATALOG)
print("Selection index:", SELECTION)


Catalog: /global/cfs/cdirs/desi/spectro/redux/loa/zcatalog/v1/zall-pix-loa.fits
Selection index: /pscratch/sd/z/zixinq/loa_selected.sqlite


## 3. Inspect the schema and a small preview
Only column names, the row count, and five rows are inspected here.


In [5]:
with fitsio.FITS(str(CATALOG)) as hdus:
    table = hdus["ZCATALOG"]
    missing = set(COLUMNS) - set(table.get_colnames())
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    catalog_row_count = table.get_nrows()
    preview = table.read(rows=np.arange(min(5, catalog_row_count)), columns=COLUMNS)
print(f"Catalog rows: {catalog_row_count:,}")
display(preview)


Catalog rows: 64,100,684


array([(39628473198710603, 'cmx', 'other', 2152,  8.04204950e-01, 9.72250670e-06,   0, 'GALAXY', 6.30432053e+03,   0,  True),
       (39628473198711006, 'cmx', 'other', 2152, -7.30834989e-05, 7.51184654e-07,   0, 'STAR', 3.97213507e+05,   0,  True),
       (39628473198711342, 'cmx', 'other', 2152,  8.47712390e-04, 4.78693071e-48, 514, 'GALAXY', 1.94266889e+84, 512,  True),
       (39628473198711925, 'cmx', 'other', 2152,  3.90711512e-01, 2.01467563e-05,   0, 'GALAXY', 3.15751993e+03,   0,  True),
       (39628473202901028, 'cmx', 'other', 2152,  6.65839320e-01, 7.94495483e-05,   0, 'GALAXY', 1.76319753e+02,   0,  True)],
      dtype=[('TARGETID', '>i8'), ('SURVEY', '<U7'), ('PROGRAM', '<U6'), ('HEALPIX', '>i4'), ('Z', '>f8'), ('ZERR', '>f8'), ('ZWARN', '>i8'), ('SPECTYPE', '<U6'), ('DELTACHI2', '>f8'), ('COADD_FIBERSTATUS', '>i4'), ('ZCAT_PRIMARY', '?')])

## 4. Define the cuts and exact coadd mapping
A target is extracted only from the survey/program/HEALPix of its primary catalog row. A global list of target IDs alone would also match nonprimary observations.


In [6]:
def text(value):
    return value.decode().strip() if isinstance(value, bytes) else str(value).strip()

def select_mask(rows, types, surveys, min_deltachi2):
    keep = (rows['ZCAT_PRIMARY'].astype(bool)
            & np.isin(np.char.strip(rows['SPECTYPE'].astype('U')), types)
            & (rows['ZWARN'] == 0) & (rows['COADD_FIBERSTATUS'] == 0)
            & np.isfinite(rows['Z']) & (rows['Z'] > -1)
            & (rows['TARGETID'] > 0))
    if surveys:
        keep &= np.isin(np.char.strip(rows['SURVEY'].astype('U')), surveys)
    if min_deltachi2 is not None:
        keep &= np.isfinite(rows['DELTACHI2']) & (rows['DELTACHI2'] >= min_deltachi2)
    return keep

def coadd_relative(row):
    survey, program, pix = text(row['SURVEY']), text(row['PROGRAM']), int(row['HEALPIX'])
    if '/' in survey + program or not survey or not program or pix < 0:
        raise ValueError('Invalid catalog location')
    return f'{survey}/{program}/{pix // 100}/{pix}/coadd-{survey}-{program}-{pix}.fits'


## 5. Build the reusable index
The following function streams the catalog and uses SQLite to group targets by coadd. Its unique TARGETID constraint rejects duplicate primary rows. A temporary file is renamed only after the index is complete. No flux arrays are read.

An existing complete index with identical settings is reused; incompatible settings raise an error rather than overwriting it.


In [7]:
def prepare(args):
    if args.selection.exists():
        raise FileExistsError(f'{args.selection} already exists; use a new index path')
    import fitsio
    args.selection.parent.mkdir(parents=True, exist_ok=True)
    tmp = args.selection.with_name(args.selection.name + f'.{uuid.uuid4().hex}.tmp')
    settings = dict(catalog=str(args.catalog.resolve()), types=args.types,
                    surveys=args.surveys, min_deltachi2=args.min_deltachi2,
                    primary=True, zwarn=0, coadd_fiberstatus=0)
    count = 0
    try:
        with sqlite3.connect(tmp) as db, fitsio.FITS(str(args.catalog)) as hdus:
            table = hdus['ZCATALOG']
            missing = set(COLUMNS) - set(table.get_colnames())
            if missing:
                raise ValueError(f'Missing catalog columns: {sorted(missing)}')
            db.execute('CREATE TABLE metadata (key TEXT PRIMARY KEY, value TEXT)')
            db.execute('''CREATE TABLE targets (targetid INTEGER PRIMARY KEY,
                       coadd TEXT NOT NULL, z REAL, zerr REAL, zwarn INTEGER,
                       deltachi2 REAL, spectype TEXT)''')
            for start in range(0, table.get_nrows(), args.chunk_rows):
                stop = min(start + args.chunk_rows, table.get_nrows())
                rows = table.read(rows=np.arange(start, stop), columns=COLUMNS)
                chosen = rows[select_mask(rows, args.types, args.surveys, args.min_deltachi2)]
                records = [(int(r['TARGETID']), coadd_relative(r), float(r['Z']),
                            float(r['ZERR']), int(r['ZWARN']), float(r['DELTACHI2']),
                            text(r['SPECTYPE'])) for r in chosen]
                # Duplicate primary TARGETIDs are an error, never silently dropped.
                db.executemany('INSERT INTO targets VALUES (?,?,?,?,?,?,?)', records)
                db.commit()
                count += len(records)
                print(f'catalog_rows={stop:,} selected={count:,}', flush=True)
            if count == 0:
                raise ValueError('Selection is empty')
            db.execute('CREATE INDEX by_coadd ON targets(coadd)')
            db.executemany('INSERT INTO metadata VALUES (?,?)', [
                ('selection_id', uuid.uuid4().hex), ('settings', json.dumps(settings)),
                ('complete', 'true')])
            db.commit()
        os.replace(tmp, args.selection)
    except BaseException:
        tmp.unlink(missing_ok=True)
        raise
    print(f'Selection ready: {args.selection}; targets={count:,}', flush=True)


In [10]:
settings = dict(catalog=str(CATALOG.resolve()), types=TYPES,
                surveys=SURVEYS, min_deltachi2=MIN_DELTACHI2,
                primary=True, zwarn=0, coadd_fiberstatus=0)

if SELECTION.exists():
    with sqlite3.connect(SELECTION.resolve().as_uri() + "?mode=ro", uri=True) as db:
        saved = dict(db.execute("SELECT key,value FROM metadata"))
    if saved.get("complete") != "true" or json.loads(saved["settings"]) != settings:
        raise ValueError("Existing index is incomplete or uses different cuts; choose a new SELECTION path")
    print("Reusing completed index:", SELECTION)
else:
    prepare(argparse.Namespace(
        catalog=CATALOG, selection=SELECTION, types=TYPES, surveys=SURVEYS,
        min_deltachi2=MIN_DELTACHI2, chunk_rows=CHUNK_ROWS,
    ))


catalog_rows=200,000 selected=160,129
catalog_rows=400,000 selected=324,948
catalog_rows=600,000 selected=483,641
catalog_rows=800,000 selected=616,864
catalog_rows=1,000,000 selected=746,616
catalog_rows=1,200,000 selected=900,826
catalog_rows=1,400,000 selected=1,057,833
catalog_rows=1,600,000 selected=1,211,013
catalog_rows=1,800,000 selected=1,359,057
catalog_rows=2,000,000 selected=1,511,372
catalog_rows=2,200,000 selected=1,642,987
catalog_rows=2,400,000 selected=1,767,881
catalog_rows=2,600,000 selected=1,903,543
catalog_rows=2,800,000 selected=2,062,907
catalog_rows=3,000,000 selected=2,225,761
catalog_rows=3,200,000 selected=2,378,653
catalog_rows=3,400,000 selected=2,511,758
catalog_rows=3,600,000 selected=2,661,587
catalog_rows=3,800,000 selected=2,817,029
catalog_rows=4,000,000 selected=2,981,956
catalog_rows=4,200,000 selected=3,148,825
catalog_rows=4,400,000 selected=3,307,319
catalog_rows=4,600,000 selected=3,470,936
catalog_rows=4,800,000 selected=3,623,727
catalog_rows

## 6. Check selected counts
Counts refer to the catalog selection before any pixel-validity filtering during extraction. The primary key guarantees a single selected row per TARGETID. Tables are used here for exact counts; no population counts are assumed in advance.


In [11]:
with sqlite3.connect(SELECTION.resolve().as_uri() + "?mode=ro", uri=True) as db:
    selected_count = db.execute("SELECT COUNT(*) FROM targets").fetchone()[0]
    coadd_count = db.execute("SELECT COUNT(DISTINCT coadd) FROM targets").fetchone()[0]
    class_counts = db.execute("SELECT spectype,COUNT(*) FROM targets GROUP BY spectype ORDER BY spectype").fetchall()
    survey_counts = db.execute("SELECT substr(coadd,1,instr(coadd,'/')-1),COUNT(*) FROM targets GROUP BY 1 ORDER BY 1").fetchall()
    sample = db.execute("SELECT targetid,coadd,z,spectype FROM targets ORDER BY targetid LIMIT 10").fetchall()

assert 0 < selected_count <= catalog_row_count
assert sum(count for _, count in class_counts) == selected_count
print(f"Selected spectra: {selected_count:,} / {catalog_row_count:,} ({selected_count/catalog_row_count:.2%})")
print(f"Coadd files with selected spectra: {coadd_count:,}")

def count_table(label, rows):
    display(Markdown(f"| {label} | Selected spectra |\n|---|---:|\n" +
                     "\n".join(f"| {name} | {count:,} |" for name, count in rows)))

count_table("Redrock class", class_counts)
count_table("Survey", survey_counts)
display(sample)


Selected spectra: 45,959,907 / 64,100,684 (71.70%)
Coadd files with selected spectra: 56,519


| Redrock class | Selected spectra |
|---|---:|
| GALAXY | 30,869,279 |
| QSO | 2,835,775 |
| STAR | 12,254,853 |

| Survey | Selected spectra |
|---|---:|
| cmx | 3,163 |
| main | 43,934,769 |
| special | 502,351 |
| sv1 | 502,124 |
| sv2 | 73,133 |
| sv3 | 944,367 |

[(6448025174016,
  'sv1/dark/284/28478/coadd-sv1-dark-28478.fits',
  0.02219053246998218,
  'GALAXY'),
 (6521555517440,
  'sv1/dark/104/10436/coadd-sv1-dark-10436.fits',
  0.010016011261740405,
  'GALAXY'),
 (6536638234624,
  'sv1/dark/285/28534/coadd-sv1-dark-28534.fits',
  0.07640640183971818,
  'GALAXY'),
 (6546033475584,
  'sv1/dark/50/5066/coadd-sv1-dark-5066.fits',
  0.026939032735136263,
  'GALAXY'),
 (45650485968998,
  'sv1/other/42/4298/coadd-sv1-other-4298.fits',
  6.322331906684941e-05,
  'STAR'),
 (45650490163336,
  'sv1/other/42/4298/coadd-sv1-other-4298.fits',
  0.0001148266761439759,
  'STAR'),
 (45650490163341,
  'sv1/other/42/4298/coadd-sv1-other-4298.fits',
  0.0001185611676766916,
  'STAR'),
 (45786075234314,
  'sv1/other/279/27975/coadd-sv1-other-27975.fits',
  0.00018834976985143437,
  'STAR'),
 (45803343183883,
  'sv1/other/279/27981/coadd-sv1-other-27981.fits',
  9.214874988986514e-05,
  'GALAXY'),
 (45803351578487,
  'sv1/other/279/27981/coadd-sv1-other-27981.fi

## 7. Spectra Extraction
This cell prints a pilot command; it does not start extraction. Run the printed command in your working DESI compute environment with `read_loa_selected.py` uploaded. The 80% valid-pixel cut is applied there. After that pilot, review the retained counts before a full run.

The SQLite file contains the selected IDs, exact coadd paths, and redshift metadata; it does not contain spectra. Keep it for subsequent extraction jobs. Normalization and training splits are later steps.


In [8]:
import shlex
EXPORTER = Path("/pscratch/sd/z/zixinq/read_loa_selected.py")
DESTINATION = Path("/global/cfs/cdirs/desi/users/zixinq/data/dr2_catalog/loa_good_spectra")
command = [sys.executable, str(EXPORTER), "extract", "--selection", str(SELECTION),
           "--destination", str(DESTINATION), "--min-valid-fraction", "0.8", "--limit", "1"]
print("export HDF5_USE_FILE_LOCKING=FALSE")
print(shlex.join(command))


export HDF5_USE_FILE_LOCKING=FALSE
/global/common/software/nersc/pe/conda-envs/26.8.1/python-3.13/nersc-python/bin/python /pscratch/sd/z/zixinq/read_loa_selected.py extract --selection /pscratch/sd/z/zixinq/loa_good.sqlite --destination /global/cfs/cdirs/desi/users/zixinq/data/dr2_catalog/loa_good_spectra --min-valid-fraction 0.8 --limit 1
